# Asia’s Carbon Transition
### Spatial patterns in emissions, energy and development, 1990–2023

A portfolio project using **50 UN M49 Asian economies**, 34 years, and 17 source-unit variables. Start with a **2023 cross-section** before trying the optional balanced spatial panel.

**How to run:** open this notebook in [Google Colab](https://colab.research.google.com/), run the cells in order, and upload **only `Asia_Climate_Spatial_Panel_1990_2023.xlsx`** when prompted. No API key is required. The workbook supplies coordinates; a separate shapefile is unnecessary for regression.

The workbook preserves raw values. This notebook creates logs and model-specific exclusions in memory. The source dataset and preprocessing/weights were checked locally. The PySAL estimation cells are provided for execution in Colab; no fitted model results are pre-populated.


## 1. Question and scope

**Portfolio question:** Do Asian economies with similar geographic neighbourhoods show similar carbon-emission patterns, and how are those patterns associated with renewable energy, energy intensity and economic development?

Use `co2_pc_t` as the primary outcome. It is CO2 emissions **excluding LULUCF**, in tonnes of CO2 equivalent per person. Lower emissions indicate lower carbon pressure. They do not establish carbon neutrality, which needs a consistent emissions-and-removals balance. GHG totals including land use are available for a separate descriptive exercise.

This project covers mitigation aspects of SDG 13, with links to energy and forest indicators. It does not evaluate every SDG 13 target. EDGAR/WDI emissions are a different compilation from the UNFCCC reporting used for official SDG 13.2.2.


In [ ]:
%pip -q install libpysal spreg folium openpyxl


In [ ]:
from pathlib import Path
import io, json, shutil, importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse.csgraph import connected_components
from libpysal.weights import W
import spreg
import folium
from IPython.display import display

SEED = 2026
PERMUTATIONS = 999
YEAR = 2023
K_NEIGHBOURS = 4
PANEL_FIRST_YEAR, PANEL_LAST_YEAR = 2000, 2023
RUN_PANEL = False  # Set True after completing the cross-sectional workflow.
EXCLUDE_ISO3 = []  # E.g. ['HKG','MAC'] for a separate sensitivity analysis.

Y_SOURCE = 'co2_pc_t'
X_SOURCE = ['renewable_energy_pct', 'energy_intensity_mj_2021ppp',
            'gdp_pc_2015_usd', 'urban_population_pct', 'industry_value_added_pct']
LOG_X = {'gdp_pc_2015_usd'}
LOG_Y = True

RESULTS = Path('/content/asia_climate_results')
RESULTS.mkdir(parents=True, exist_ok=True)
versions = {p: metadata.version(p) for p in ['numpy','pandas','scipy','libpysal','spreg','folium']}
(RESULTS/'package_versions.json').write_text(json.dumps(versions, indent=2))
print(versions)


In [ ]:
from google.colab import files
uploaded = files.upload()
workbooks = [name for name in uploaded if name.lower().endswith('.xlsx')]
if len(workbooks) != 1:
    raise ValueError('Upload exactly one Excel workbook: Asia_Climate_Spatial_Panel_1990_2023.xlsx')
FILE = workbooks[0]
panel = pd.read_excel(io.BytesIO(uploaded[FILE]), sheet_name='Panel_Data')
geography = pd.read_excel(io.BytesIO(uploaded[FILE]), sheet_name='Country_Geography', dtype={'m49_code':str})
dictionary = pd.read_excel(io.BytesIO(uploaded[FILE]), sheet_name='Variable_Dictionary')
assert not panel.duplicated(['iso3','year']).any(), 'Duplicate economy-year keys.'
assert not geography.iso3.duplicated().any(), 'Duplicate geographic keys.'
assert panel.year.min() == 1990 and panel.year.max() == 2023
assert len(panel) == 1700 and panel.iso3.nunique() == 50
assert set(panel.iso3) == set(geography.iso3)
print(f'{len(panel):,} rows, {panel.iso3.nunique()} economies, {panel.year.nunique()} years')
display(dictionary[['header','unit','role','source_code']])


## 2. Check data availability and source anomalies

The table is rectangular but observations are incomplete. Blank values remain missing, not zero. Renewable-electricity data end in 2021. Renewable-capacity and GHG-including-LULUCF data begin in 2000 in this extract. UN renewable-consumption and energy-intensity releases extend through 2023.

The UN energy-intensity observation `basePeriod` and series description specify **2021 PPP**, consistent with the official metadata. Its API unit-code dictionary still has an outdated 2011 label. The source values have not been converted.

The source publishes two negative fossil-energy shares: Kuwait and Singapore in 2015. These are retained in the workbook. The baseline model does not use that variable; the preparation function below explicitly excludes out-of-range shares if one is selected. Published trade/GDP values above 100 are valid.


In [ ]:
variable_headers = dictionary.header.tolist()
availability = panel[variable_headers].notna().mean().sort_values()
display((availability*100).round(1).rename('available_percent').to_frame())
fig, ax = plt.subplots(figsize=(10,6))
ax.barh(availability.index, availability.values*100, color='#315C84')
ax.set(xlabel='Available observations (%)', title='Coverage of Asian economy-year observations, 1990–2023', xlim=(0,105))
ax.spines[['top','right']].set_visible(False)
fig.tight_layout(); fig.savefig(RESULTS/'01_data_availability.png', dpi=200, bbox_inches='tight'); plt.show()

share_headers = ['renewable_energy_pct','renewable_electricity_pct','fossil_energy_pct',
                 'forest_area_pct','electricity_access_pct','urban_population_pct']
quality_flags=[]
for h in share_headers:
    q=panel.loc[panel[h].notna() & ~panel[h].between(0,100), ['iso3','year',h]].copy()
    if not q.empty:
        q=q.rename(columns={h:'source_value'}); q['header']=h; quality_flags.append(q)
flags=pd.concat(quality_flags,ignore_index=True) if quality_flags else pd.DataFrame()
display(flags)
flags.to_csv(RESULTS/'source_quality_flags.csv',index=False)


In [ ]:
# A constant sample avoids confusing changing country coverage with changing emissions.
valid_codes = (panel.groupby('iso3')['co2_pc_t'].count() == 34)
fixed_codes = valid_codes[valid_codes].index
trajectory = panel[panel.iso3.isin(fixed_codes)].copy()
annual = trajectory.groupby('year')['co2_pc_t'].agg(['median', lambda z:z.quantile(.25), lambda z:z.quantile(.75)])
annual.columns=['median','q25','q75']
fig, ax=plt.subplots(figsize=(10,4.5))
ax.fill_between(annual.index,annual.q25,annual.q75,color='#C8D9E8',label='Interquartile range')
ax.plot(annual.index,annual['median'],color='#244E76',linewidth=2,label='Median economy')
ax.set(xlabel='Year',ylabel='CO2 excluding LULUCF (t CO2e per capita)',title=f'Carbon-pressure trajectory across {len(fixed_codes)} consistently observed economies')
ax.legend(frameon=False); ax.spines[['top','right']].set_visible(False)
fig.tight_layout(); fig.savefig(RESULTS/'02_emissions_trajectory.png',dpi=200,bbox_inches='tight'); plt.show()


## 3. Prepare one cross-section and its spatial weights

The baseline uses five explanatory variables rather than placing every collected variable in one regression. CO2 totals, CO2 per capita and GHG totals are **alternative outcomes**, not a set of simultaneous predictors.

`log(CO2 per capita)` and `log(GDP per capita)` are created here only. Predictors are standardised on the estimation sample for stable fitting, so coefficients and impacts correspond to a **one-standard-deviation** change in each transformed predictor. Percentages are first kept in their original percentage-point units. No interpolation or winsorisation is used.

Coordinates are **Natural Earth country label points**, not country centroids or capital cities. Distances are great-circle distances. Each economy selects its k nearest neighbours; reciprocal links are then added. The binary graph is symmetric, while row-standardised weights need not be symmetric. All weights link economies within the selected Asian sample, so external neighbours are absent.

This is a geographic economic-association model, not a model of atmospheric CO2 transport. Report sensitivity to k, point choice, and inclusion of Hong Kong and Macao.


In [ ]:
def prepare_frame(frame):
    needed=list(dict.fromkeys([Y_SOURCE]+X_SOURCE))
    z=frame.copy()
    for c in needed:
        z[c]=pd.to_numeric(z[c], errors='coerce')
    valid=z[needed].notna().all(axis=1)
    if LOG_Y:
        valid &= z[Y_SOURCE] > 0
    for c in LOG_X:
        if c in X_SOURCE:
            valid &= z[c] > 0
    for c in set(needed).intersection(share_headers):
        valid &= z[c].between(0,100)
    valid &= ~z.iso3.isin(EXCLUDE_ISO3)
    excluded=z.loc[~valid,['iso3','country','year']+needed].copy()
    z=z.loc[valid].copy()
    z['model_y']=np.log(z[Y_SOURCE]) if LOG_Y else z[Y_SOURCE]
    model_names=[]
    for c in X_SOURCE:
        name='ln_'+c if c in LOG_X else c
        z[name]=np.log(z[c]) if c in LOG_X else z[c]
        model_names.append(name)
    return z, excluded, model_names

def make_weights(frame, k=4):
    frame=frame.sort_values('iso3').reset_index(drop=True)
    ids=frame.iso3.tolist()
    n=len(ids)
    if not 1 <= k < n:
        raise ValueError('k must be between 1 and n-1.')
    lat=np.radians(frame.latitude.to_numpy(float)); lon=np.radians(frame.longitude.to_numpy(float))
    dlat=lat[:,None]-lat[None,:]; dlon=lon[:,None]-lon[None,:]
    a=np.sin(dlat/2)**2+np.cos(lat[:,None])*np.cos(lat[None,:])*np.sin(dlon/2)**2
    distance=6371.0088*2*np.arcsin(np.sqrt(np.clip(a,0,1)))
    np.fill_diagonal(distance,np.inf)
    adjacency=np.zeros((n,n),dtype=int)
    nearest=np.argsort(distance,axis=1,kind='stable')[:,:k]
    adjacency[np.arange(n)[:,None],nearest]=1
    adjacency=np.maximum(adjacency,adjacency.T)
    neighbours={ids[i]:[ids[j] for j in np.flatnonzero(adjacency[i])] for i in range(n)}
    w=W(neighbours,id_order=ids,silence_warnings=True); w.transform='r'
    matrix=w.full()[0]
    assert w.id_order == ids
    assert np.allclose(matrix.sum(axis=1),1) and np.allclose(np.diag(matrix),0)
    assert not w.islands
    n_components=connected_components(adjacency,directed=False,return_labels=False)
    return w, matrix, distance, n_components

sample, exclusions, model_names=prepare_frame(panel.loc[panel.year==YEAR])
sample=sample.merge(geography[['iso3','longitude','latitude']],on='iso3',validate='one_to_one').sort_values('iso3').reset_index(drop=True)
print(f'Year {YEAR}: {len(sample)} included economies, {len(exclusions)} exclusions')
display(exclusions)
exclusions.to_csv(RESULTS/'cross_section_exclusions.csv',index=False)
if len(sample)<max(20,4*len(X_SOURCE)):
    raise ValueError('Too few economies for this specification. Reduce predictors or choose another year.')
w, W_matrix, distances, components=make_weights(sample,K_NEIGHBOURS)
print('Connected components:',components,'; islands:',w.islands)
means=sample[model_names].mean(); scales=sample[model_names].std(ddof=0)
if (scales<=1e-10).any():
    raise ValueError('A predictor is constant. Remove it from X_SOURCE.')
X=((sample[model_names]-means)/scales).to_numpy(float)
y=sample[['model_y']].to_numpy(float)
assert np.linalg.matrix_rank(np.column_stack([np.ones(len(X)),X]))==X.shape[1]+1
pd.DataFrame({'mean':means,'sd':scales}).to_csv(RESULTS/'predictor_scaling.csv')
pd.DataFrame(W_matrix,index=w.id_order,columns=w.id_order).to_csv(RESULTS/'weights_matrix.csv')
sample.to_csv(RESULTS/'cross_section_estimation_sample.csv',index=False)


In [ ]:
spatial_map=folium.Map(location=[30,85],zoom_start=3,tiles='CartoDB positron')
levels=sample.model_y.to_numpy(); lower,upper=levels.min(),levels.max()
from branca.colormap import linear
colours=linear.YlOrRd_09.scale(float(lower),float(upper))
colours.caption=f'{YEAR}: log CO2 per capita' if LOG_Y else f'{YEAR}: {Y_SOURCE}'
for i,row in sample.iterrows():
    for j in range(i+1,len(sample)):
        if W_matrix[i,j] > 0:
            folium.PolyLine([[row.latitude,row.longitude],[sample.iloc[j].latitude,sample.iloc[j].longitude]],color='#6B7F91',weight=1,opacity=.35).add_to(spatial_map)
    folium.CircleMarker([row.latitude,row.longitude],radius=5,color=colours(float(row.model_y)),fill=True,fill_opacity=.9,
                        tooltip=f"{row.country}: {row.co2_pc_t:.2f} t CO2e/capita").add_to(spatial_map)
colours.add_to(spatial_map)
spatial_map.save(str(RESULTS/'03_neighbour_map.html'))
display(spatial_map)


## 4. Global Moran’s I

This cell uses an explicit permutation test so the tail convention is clear. The two-sided pseudo p-value compares `|I − E[I]|` with its permutation distribution and uses the finite-permutation correction. The weight graph is held fixed. Spatial clustering in emissions does not by itself demonstrate cross-border causation.


In [ ]:
def moran_permutation(values, matrix, permutations=999, seed=2026):
    z=np.asarray(values,float).reshape(-1); z=z-z.mean(); n=len(z)
    if np.dot(z,z)<=0:
        raise ValueError('Moran I needs variation in the outcome.')
    multiplier=n/matrix.sum()
    observed=float(multiplier*(z@matrix@z)/(z@z))
    expected=-1/(n-1)
    rng=np.random.default_rng(seed)
    simulated=np.empty(permutations)
    for b in range(permutations):
        zp=rng.permutation(z)
        simulated[b]=multiplier*(zp@matrix@zp)/(zp@zp)
    p=(1+np.sum(np.abs(simulated-expected)>=abs(observed-expected)))/(permutations+1)
    return {'I':observed,'expected_I':expected,'p_two_sided':float(p),'permutations':permutations},simulated

moran_result, moran_sim=moran_permutation(y,W_matrix,PERMUTATIONS,SEED)
print(moran_result)
pd.DataFrame([moran_result]).to_csv(RESULTS/'global_moran.csv',index=False)
z=(y.ravel()-y.mean())/y.std(ddof=0)
wz=W_matrix@z
fig,ax=plt.subplots(figsize=(6,6)); ax.scatter(z,wz,color='#315C84')
line=np.linspace(z.min(),z.max(),100); ax.plot(line,moran_result['I']*line,color='#AC513A')
ax.axhline(0,color='grey',lw=.7); ax.axvline(0,color='grey',lw=.7)
ax.set(xlabel='Standardised log CO2 per capita' if LOG_Y else 'Standardised outcome',ylabel='Spatial lag of standardised outcome',
       title=f"Moran scatterplot, {YEAR}: I={moran_result['I']:.3f}")
fig.tight_layout(); fig.savefig(RESULTS/'04_moran_scatter.png',dpi=200,bbox_inches='tight'); plt.show()


## 5. OLS baseline, spatial lag (SAR), and spatial error (SEM)

OLS provides the comparison model and residual spatial diagnostics. The spatial-lag model includes a weighted neighbour outcome. The spatial-error model allows spatial correlation in unobserved disturbances. Compare AIC only for models using the same outcome, sample and explanatory variables.

`spreg.OLS` adds the constant internally. Its classical LM diagnostics are a starting point, not a causal identification test. SAR and SEM maximum-likelihood standard errors rely on their model assumptions. A small p-value, low AIC, or preferred coefficient sign should not be the sole reason for choosing a model. A small sample of around 46 economies supports a compact specification.


In [ ]:
# Collinearity report, with predictors in their transformed and standardised form.
vif=[]
for j,name in enumerate(model_names):
    other=np.column_stack([np.ones(len(X)),np.delete(X,j,axis=1)])
    fitted=other@np.linalg.lstsq(other,X[:,j],rcond=None)[0]
    r2=1-np.sum((X[:,j]-fitted)**2)/np.sum((X[:,j]-X[:,j].mean())**2)
    vif.append({'variable':name,'VIF':np.inf if r2>=1-1e-12 else 1/(1-r2)})
display(pd.DataFrame(vif)); pd.DataFrame(vif).to_csv(RESULTS/'vif.csv',index=False)

ols=spreg.OLS(y,X,w=w,spat_diag=True,moran=True,name_y='ln_co2_pc' if LOG_Y else Y_SOURCE,name_x=model_names)
sar=spreg.ML_Lag(y,X,w=w,method='full',name_y='ln_co2_pc' if LOG_Y else Y_SOURCE,name_x=model_names)
sem=spreg.ML_Error(y,X,w=w,method='full',name_y='ln_co2_pc' if LOG_Y else Y_SOURCE,name_x=model_names)
models={'OLS':ols,'SAR':sar,'SEM':sem}
comparison=[]; coefficient_tables=[]
for name,model in models.items():
    print(model.summary)
    (RESULTS/f'{name}_summary.txt').write_text(str(model.summary))
    comparison.append({'model':name,'n':len(y),'AIC':float(model.aic),'log_likelihood':float(model.logll),
                       'spatial_parameter':float(model.rho) if name=='SAR' else float(model.lam) if name=='SEM' else np.nan})
    terms=['CONSTANT']+model_names+(['rho'] if name=='SAR' else ['lambda'] if name=='SEM' else [])
    coef=np.asarray(model.betas).ravel(); se=np.asarray(model.std_err).ravel()
    assert len(terms)==len(coef)==len(se)
    coefficient_tables.append(pd.DataFrame({'model':name,'term':terms,'coefficient':coef,'std_error':se}))
comparison=pd.DataFrame(comparison); display(comparison)
comparison.to_csv(RESULTS/'model_comparison.csv',index=False)
pd.concat(coefficient_tables,ignore_index=True).to_csv(RESULTS/'model_coefficients.csv',index=False)


## 6. Interpret SAR impacts and test weights sensitivity

In a SAR model, a coefficient is not the full marginal effect because changes propagate through the weight matrix. The matrix for predictor j is `(I − rho W)^(-1) beta_j`. Its average diagonal is the direct impact; its average row sum is the total impact; the difference is the indirect impact. The table below contains point estimates only, not impact confidence intervals.

For the default log outcome and standardised predictors, impacts are changes in **log CO2 per capita per one standard deviation of a transformed predictor**. Avoid claiming causal emissions reductions. Holding the sample fixed when comparing k keeps weights sensitivity interpretable. The optional exclusion of Hong Kong/Macao changes the sample, so report it as a separate analysis.


In [ ]:
S=np.linalg.inv(np.eye(len(y))-float(sar.rho)*W_matrix)
impacts=[]
for j,name in enumerate(model_names,start=1):
    impact=S*float(np.asarray(sar.betas).ravel()[j])
    direct=float(np.trace(impact)/len(y)); total=float(impact.sum(axis=1).mean())
    impacts.append({'predictor':name,'direct':direct,'indirect':total-direct,'total':total})
impacts=pd.DataFrame(impacts); display(impacts)
impacts.to_csv(RESULTS/'sar_impacts_point_estimates.csv',index=False)

sensitivity=[]
for k in [3,4,5]:
    wk,matrix_k,_,components_k=make_weights(sample,k)
    fit=spreg.ML_Lag(y,X,w=wk,method='full',name_y='model_y',name_x=model_names)
    m,_=moran_permutation(y,matrix_k,PERMUTATIONS,SEED)
    sensitivity.append({'k':k,'n':len(y),'components':components_k,'rho':float(fit.rho),'AIC':float(fit.aic),
                        'Moran_I':m['I'],'Moran_p_two_sided':m['p_two_sided']})
sensitivity=pd.DataFrame(sensitivity); display(sensitivity)
sensitivity.to_csv(RESULTS/'weights_sensitivity.csv',index=False)


## 7. Optional balanced spatial panel

Set `RUN_PANEL=True` in the settings cell and rerun from there. The default 2000–2023 baseline has **40 complete economies and 960 observations**, with no imputation. A 1990–2023 version is possible using its smaller complete sample (31 economies for this baseline).

Each year uses exactly the same economies and W. Rows are stacked **year first, then the exact economy order in W**. This is the long-form convention expected by `Panel_FE_Lag`. Country fixed effects are handled by the estimator; year indicators are included explicitly. The estimator is a static spatial panel, not a dynamic model with a lagged own outcome. Current package versions may flag this class as a legacy interface; keep the printed version record and consult the linked documentation.

The panel section estimates a model but does not establish causal identification or solve omitted-variable bias, endogeneity, country-specific slopes, non-stationarity, or every form of temporal dependence. Compare its estimates with the cross-sectional story without assuming either model proves the mechanisms.


In [ ]:
if RUN_PANEL:
    period=panel.loc[panel.year.between(PANEL_FIRST_YEAR,PANEL_LAST_YEAR)].copy()
    prepared,excluded_panel,panel_names=prepare_frame(period)
    years=list(range(PANEL_FIRST_YEAR,PANEL_LAST_YEAR+1)); T=len(years)
    complete_counts=prepared.groupby('iso3').year.nunique()
    balanced_ids=sorted(complete_counts[complete_counts==T].index)
    balanced=prepared.loc[prepared.iso3.isin(balanced_ids)].copy()
    if len(balanced_ids)<20 or T<3:
        raise ValueError('Balanced sample is too small for the selected specification.')
    coords=geography.loc[geography.iso3.isin(balanced_ids)].sort_values('iso3').reset_index(drop=True)
    wp,wp_matrix,_,panel_components=make_weights(coords,K_NEIGHBOURS)
    assert wp.id_order==balanced_ids
    balanced=balanced.set_index(['year','iso3']).reindex(pd.MultiIndex.from_product([years,wp.id_order],names=['year','iso3'])).reset_index()
    assert not balanced[['model_y']+panel_names].isna().any().any()
    assert np.array_equal(balanced.iso3.to_numpy(),np.tile(wp.id_order,T))
    x_base=balanced[panel_names]
    means_p=x_base.mean(); scales_p=x_base.std(ddof=0)
    if (scales_p<=1e-10).any(): raise ValueError('Constant panel predictor.')
    x_scaled=((x_base-means_p)/scales_p).to_numpy(float)
    year_effects=pd.get_dummies(balanced.year,prefix='year',dtype=float).iloc[:,1:]
    xp=np.column_stack([x_scaled,year_effects.to_numpy()])
    yp=balanced[['model_y']].to_numpy(float)
    # Confirm full rank after the country within transformation, including year indicators.
    n=len(wp.id_order)
    xp_cube=xp.reshape(T,n,-1)
    xp_within=(xp_cube-xp_cube.mean(axis=0,keepdims=True)).reshape(T*n,-1)
    if np.linalg.matrix_rank(xp_within)<xp.shape[1]:
        raise ValueError('Within-transformed predictors are collinear. Revise X_SOURCE.')
    panel_fit=spreg.Panel_FE_Lag(yp,xp,w=wp,name_y='model_y',name_x=panel_names+year_effects.columns.tolist())
    print(f'{n} economies x {T} years = {len(yp)} observations. Components: {panel_components}')
    print(panel_fit.summary)
    balanced.to_csv(RESULTS/'balanced_panel_sample.csv',index=False)
    excluded_panel.to_csv(RESULTS/'panel_invalid_rows.csv',index=False)
    pd.DataFrame({'iso3':sorted(set(period.iso3)-set(balanced_ids))}).to_csv(RESULTS/'panel_excluded_economies.csv',index=False)
    (RESULTS/'panel_fe_sar_summary.txt').write_text(str(panel_fit.summary))
else:
    print('Optional panel skipped. Set RUN_PANEL=True to estimate it.')


## 8. Present the portfolio and download the outputs

Use the title **“Asia’s Carbon Transition: A Spatial Analysis of Emissions, Energy and Development.”** A useful project page contains:

1. The question, 50-economy source scope, 1990–2023 period, and a short data dictionary.
2. Data coverage and the documented source anomalies.
3. A fixed-sample emissions trajectory, a geographic neighbour map and the Moran scatterplot.
4. The same-sample OLS/SAR/SEM comparison, diagnostics, SAR impacts and k sensitivity.
5. Three evidence-based observations written only after reviewing the fitted results.
6. Limitations: upstream estimates, missingness, coordinate assumptions, traditional biomass, small samples, source revisions, and association versus causation.

Avoid calling this a carbon-neutrality score or an official SDG assessment. Do not select predictors, economies or weights solely to obtain significance or preferred signs. Keep all source units in the published dataset, and document modelling transformations separately.


In [ ]:
settings={'year':YEAR,'k':K_NEIGHBOURS,'permutations':PERMUTATIONS,'seed':SEED,
          'outcome':Y_SOURCE,'log_y':LOG_Y,'predictors':X_SOURCE,'log_x':sorted(LOG_X),
          'excluded_economies':EXCLUDE_ISO3,'panel_first_year':PANEL_FIRST_YEAR,
          'panel_last_year':PANEL_LAST_YEAR,'run_panel':RUN_PANEL,
          'cross_section_n':len(sample),'coordinate_type':'Natural Earth country label points'}
(RESULTS/'analysis_settings.json').write_text(json.dumps(settings,indent=2))
archive=shutil.make_archive('/content/Asia_Climate_Spatial_Results','zip',RESULTS)
files.download(archive)


## Sources and technical references

- [UN M49 geographic scope](https://unstats.un.org/unsd/methodology/m49/)
- [World Bank CO2 per capita series](https://data.worldbank.org/indicator/EN.GHG.CO2.PC.CE.AR5)
- [UN SDG renewable-energy metadata, 7.2.1](https://unstats.un.org/sdgs/metadata/files/Metadata-07-02-01.pdf)
- [UN SDG energy-intensity metadata, 7.3.1](https://unstats.un.org/sdgs/metadata/files/Metadata-07-03-01.pdf)
- [UN SDG renewable-capacity metadata, 7.b.1](https://unstats.un.org/sdgs/metadata/files/Metadata-07-0b-01.pdf)
- [UN SDG emissions metadata, 13.2.2](https://unstats.un.org/sdgs/metadata/files/Metadata-13-02-02.pdf)
- [Natural Earth geographic source](https://www.naturalearthdata.com/downloads/50m-cultural-vectors/)
- [PySAL weight objects](https://pysal.org/libpysal/stable/generated/libpysal.weights.W.html)
- [PySAL OLS diagnostics](https://pysal.org/spreg/generated/spreg.OLS.html)
- [PySAL spatial lag](https://pysal.org/spreg/generated/spreg.ML_Lag.html)
- [PySAL spatial error](https://pysal.org/spreg/generated/spreg.ML_Error.html)
- [PySAL fixed-effects spatial panel](https://pysal.org/spreg/generated/spreg.Panel_FE_Lag.html)

The workbook's Variable_Dictionary gives an exact source code and data/metadata URL for every column. Observation_Provenance.csv in the project bundle retains observation-level upstream sources and flags. Sources were retrieved on **4 November 2026**.
